# 02 — Occupation Scores (ILO–NASK 2023 + 2025 vintages)

Build the per-individual GenAI exposure variables for the ESS panel.

## What this notebook does

1. Downloads the ILO–NASK 2025 occupation-score xlsx (Gmyrek et al. 2025, public on GitHub) into `data/raw/scores/` if absent.
2. Loads occupation-level mean scores for 2023 (GBB vintage) and 2025 (refined ILO–NASK vintage) per ISCO-08 4-digit.
3. Reads the integrated ESS panel (`data/interim/ess_full.parquet` from notebook 01).
4. Applies the analysis vintage rule: **2023 GBB scores for R6–R10, 2025 ILO–NASK scores for R11.** Adds `genai_i` and `genai_i_static` columns.
5. Persists the score table and the merged panel.
6. Diagnoses score coverage per round (which ISCO-08 codes don't appear in ILO-NASK) and visualises the **R10 → R11 vintage shift** that injects within-country time variation.

## Why a vintage rule

Webb (2020), Felten 2021/2023, and Eloundou 2024 all publish a **single** exposure score per occupation — they are time-invariant at the occupation level. The 2025 ILO–NASK paper re-scores the 2023 GBB occupation set using GPT-4 / Gemini-1.5-era methodology. Pairing R6–R10 with the 2023 vintage and R11 with the 2025 vintage is the cleanest available approximation of capability-driven within-country variation in $E_o$, beyond pure compositional drift in $w_{oct}$.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
import requests

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT.name != "MLA" and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.mla.exposure import (  # noqa: E402
    ILO_NASK_XLSX,
    SCORES_DIR,
    VINTAGE_FOR_ROUND,
    assign_vintage_score,
    load_ilo_nask_scores,
    report_score_coverage,
)

INTERIM_DIR = REPO_ROOT / "data" / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
REPO_ROOT, ILO_NASK_XLSX

(PosixPath('/Users/karlalucic/Code/coursework/KUL/2sem/MLA'),
 PosixPath('/Users/karlalucic/Code/coursework/KUL/2sem/MLA/data/raw/scores/Final_Scores_ISCO08_Gmyrek_et_al_2025.xlsx'))

## 1. Download ILO–NASK score files (idempotent)

In [2]:
ILO_NASK_BASE = (
    "https://raw.githubusercontent.com/pgmyrek/2025_GenAI_scores_ISCO08/main/"
)
to_fetch = {
    "Final_Scores_ISCO08_Gmyrek_et_al_2025.xlsx": SCORES_DIR
    / "Final_Scores_ISCO08_Gmyrek_et_al_2025.xlsx",
    "4digits_with_tasks.xlsx": SCORES_DIR / "4digits_with_tasks.xlsx",
}
SCORES_DIR.mkdir(parents=True, exist_ok=True)
for name, dest in to_fetch.items():
    if dest.exists():
        print(f"present  {dest.name}  ({dest.stat().st_size / 1e6:.1f} MB)")
        continue
    r = requests.get(ILO_NASK_BASE + name, timeout=120)
    r.raise_for_status()
    dest.write_bytes(r.content)
    print(f"fetched  {dest.name}  ({dest.stat().st_size / 1e6:.1f} MB)")

present  Final_Scores_ISCO08_Gmyrek_et_al_2025.xlsx  (2.6 MB)
present  4digits_with_tasks.xlsx  (0.2 MB)


## 2. Load occupation-level scores per ISCO-08 4-digit

In [3]:
scores = load_ilo_nask_scores()
print(f"shape: {scores.shape}")
print(f"isco08 range: {scores.isco08.min()} .. {scores.isco08.max()}")
scores.head()

shape: (427, 6)
isco08 range: 1111 .. 9629


,isco08,title,score_2023,score_2025,sd_2023,sd_2025
0,1111,Legislators,0.22,0.31,0.07,0.04
1,1112,Senior Government Officials,0.32,0.38,0.16,0.08
2,1113,Traditional Chiefs and Heads of Villages,0.33,0.23,0.19,0.10
3,1114,Senior Officials of Special-interest Organizat...,0.31,0.37,0.15,0.06
4,1120,Managing Directors and Chief Executives,0.30,0.38,0.12,0.06


In [4]:
summary = scores[["score_2023", "score_2025", "sd_2023", "sd_2025"]].describe().round(3)
summary

,score_2023,score_2025,sd_2023,sd_2025
count,427.000,427.000,427.000,427.000
mean,0.302,0.297,0.138,0.086
std,0.161,0.145,0.073,0.047
min,0.080,0.090,0.000,0.000
25%,0.160,0.180,0.080,0.050
50%,0.290,0.270,0.140,0.080
75%,0.390,0.390,0.190,0.110
max,0.770,0.700,0.360,0.230


The observed occupation-score summaries match the published ILO-NASK distribution: the 2023 mean is about 0.30, and the 2025 mean is about 0.29 with SD about 0.14.

In [5]:
# Persist the deduped occupation-level table.
scores_path = INTERIM_DIR / "occupation_scores.parquet"
scores.to_parquet(scores_path, index=False)
print(f"wrote {scores_path}  ({scores_path.stat().st_size / 1e6:.2f} MB)")

wrote /Users/karlalucic/Code/coursework/KUL/2sem/MLA/data/interim/occupation_scores.parquet  (0.02 MB)


## 3. Merge into the ESS panel under the vintage rule

In [6]:
panel = pd.read_parquet(INTERIM_DIR / "ess_full.parquet")
print(f"panel shape pre-merge: {panel.shape}")
print(f"vintage rule: {VINTAGE_FOR_ROUND}")
merged = assign_vintage_score(panel, scores)
print(f"panel shape post-merge: {merged.shape}")
merged[["essround", "isco08", "genai_i", "genai_i_static"]].head()

panel shape pre-merge: (276491, 21)
vintage rule: {6: 2023, 7: 2023, 8: 2023, 9: 2023, 10: 2023, 11: 2025}
panel shape post-merge: (276491, 23)


,essround,isco08,genai_i,genai_i_static
0,6,5414,0.17,0.20
1,6,66666,NaN,NaN
2,6,1321,0.36,0.38
3,6,66666,NaN,NaN
4,6,3135,0.41,0.31


## 4. Coverage diagnostic

Share of ESS individuals per round whose `isco08` matches an ILO-NASK occupation. Misses are typically (a) armed forces (codes starting with 0), (b) sentinel codes (66666, 99999) for refusals/missing, (c) 4-digit codes outside the 427-code ILO-NASK set.

In [7]:
cov = report_score_coverage(merged, "genai_i").round(3)
cov

,n_individuals,n_with_score,share_with_score
essround,,,
6,54673,46125,0.844
7,40185,34291,0.853
8,44387,37612,0.847
9,49519,42168,0.852
10,37611,29264,0.778
11,50116,41013,0.818


In [8]:
# Inspect the unmatched ISCO-08 codes (top 10 by frequency).
missing = merged.loc[merged["genai_i"].isna(), "isco08"].value_counts().head(10)
missing

isco08
66666    25150
99999     3182
77777     1973
5249       822
88888      772
8189       699
1439       652
2300       591
5220       454
110        355
Name: count, dtype: Int64

## 5. The R10 → R11 vintage shift

Mean of `genai_i` (vintage rule) vs. `genai_i_static` (2025 throughout) per round. The gap between them at R6–R10 is exactly the within-country variation injected by the capability shift; if it were zero, only compositional drift in $w_{oct}$ would drive within-effects.

In [9]:
rounds_means = (
    merged.groupby("essround")
    .agg(
        genai_i_mean=("genai_i", "mean"),
        genai_i_static_mean=("genai_i_static", "mean"),
    )
    .round(4)
)
rounds_means["vintage_gap"] = (
    rounds_means["genai_i_mean"] - rounds_means["genai_i_static_mean"]
)
rounds_means

,genai_i_mean,genai_i_static_mean,vintage_gap
essround,,,
6,0.3083,0.2974,0.0109
7,0.3127,0.3029,0.0098
8,0.3169,0.3054,0.0115
9,0.3131,0.3039,0.0092
10,0.3162,0.3059,0.0103
11,0.3072,0.3072,0.0000


## 6. Persist the merged panel for downstream notebooks

In [10]:
out_path = INTERIM_DIR / "ess_panel_with_scores.parquet"
merged.to_parquet(out_path, index=False)
print(f"wrote {out_path}  ({out_path.stat().st_size / 1e6:.1f} MB)")

wrote /Users/karlalucic/Code/coursework/KUL/2sem/MLA/data/interim/ess_panel_with_scores.parquet  (4.7 MB)


## 7. Occupation-score validation

The reproducible pipeline requires a primary occupation-score table with 2023 and 2025 vintages, plus individual-level `genai_i` and `genai_i_static` columns merged onto the ESS panel.

Primary `genai_i` and `genai_i_static` are joined onto the panel. Webb, Felten, and Eloundou alternatives are not implemented here because they require SOC-keyed source files and an ISCO-08 to SOC crosswalk.

In [11]:
have_genai = "genai_i" in merged.columns
have_static = "genai_i_static" in merged.columns
share_with_score = cov["share_with_score"].mean()
checkpoint = have_genai and have_static and share_with_score >= 0.7
print(f"genai_i column present:        {have_genai}")
print(f"genai_i_static column present: {have_static}")
print(f"mean per-round score coverage: {share_with_score:.1%}")
print(f"Occupation-score validation:    {'PASS' if checkpoint else 'pending'}")

genai_i column present:        True
genai_i_static column present: True
mean per-round score coverage: 83.2%
Occupation-score validation:    PASS
